# PCG-LLM — Tiny Model Training (Kaggle T4)

Trains the **Tiny PCG-LLM** (50–100M parameters) on a 1B-token slice of FineWeb-Edu + The Stack v2.

**Requirements**
- Kaggle Notebook with **2× T4 GPU** accelerator enabled
- Internet access ON (for pip/uv and HuggingFace tokenizer download)
- ~20 GB working directory space (`/kaggle/working/`)

**What this notebook does**
1. Installs `uv` and syncs all training dependencies
2. Logs in to HuggingFace (for the Llama-3 tokenizer)
3. Exports and displays the resolved `TrainingConfig`
4. Runs the training loop — checkpoints every 500 steps to `/kaggle/working/checkpoints/`
5. Resumes automatically from the latest checkpoint on kernel restart

**Monitoring**: open [wandb.ai](https://wandb.ai) in a second tab to track loss, solver steps, node variance, and EAGLE acceptance rate in real time.

---
## 0 — Environment check

In [ ]:
import subprocess, sys, os

# Confirm GPU is available
import torch
print(f"PyTorch   : {torch.__version__}")
print(f"CUDA      : {torch.version.cuda}")
print(f"GPU count : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}  : {props.name}  {props.total_memory / 1e9:.1f} GB")

# Working directory
print(f"\nWorking dir : {os.getcwd()}")
disk = subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print(disk.stdout)

---
## 1 — Install dependencies

In [ ]:
# Install uv (fast Python package manager)
!pip install uv --quiet
print("uv installed")

In [ ]:
# Clone the repo (skip if already present)
import os
REPO_DIR = "/kaggle/working/pcg-llm"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ey3lock3r/pcg-llm.git {REPO_DIR}
else:
    print(f"Repo already present at {REPO_DIR} — pulling latest")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print(f"Working in: {os.getcwd()}")

In [ ]:
# Sync all training dependencies (torch is already installed by Kaggle; uv will reuse it)
!uv sync --extra training --system
print("Dependencies synced")

---
## 2 — HuggingFace login (Llama-3 tokenizer)

The Llama-3 tokenizer is gated. Accept the license at  
[huggingface.co/meta-llama/Meta-Llama-3-8B](https://huggingface.co/meta-llama/Meta-Llama-3-8B)  
then paste your token below.

> **Skip this cell** if you have already logged in during this Kaggle session,  
> or if you prefer to use the public fallback tokenizer (slightly different vocab).

In [ ]:
# Option A: paste token directly (add your token to Kaggle Secrets instead for security)
HF_TOKEN = ""  # <- paste your HuggingFace token here, or leave blank to use Kaggle Secrets

if not HF_TOKEN:
    # Try Kaggle Secrets (add HF_TOKEN in Notebook → Add-ons → Secrets)
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle Secrets")
    except Exception:
        print("No HF_TOKEN found — will use public fallback tokenizer")

if HF_TOKEN:
    import subprocess
    result = subprocess.run(
        ["uv", "run", "huggingface-cli", "login", "--token", HF_TOKEN],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)

---
## 3 — W&B login (optional but recommended)

Tracks loss, solver convergence, node variance, and EAGLE acceptance rate in real time.  
Create a free account at [wandb.ai](https://wandb.ai), then add your API key to Kaggle Secrets as `WANDB_API_KEY`.

In [ ]:
WANDB_API_KEY = ""  # <- paste key here, or use Kaggle Secrets

if not WANDB_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
        print("WANDB_API_KEY loaded from Kaggle Secrets")
    except Exception:
        print("No WANDB_API_KEY — W&B logging disabled (training will still run)")

if WANDB_API_KEY:
    import os
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    !uv run wandb login --relogin
else:
    import os
    os.environ["WANDB_DISABLED"] = "true"

---
## 4 — Review training configuration

The `tiny` preset targets Kaggle T4 (2× 16 GB VRAM).  
Edit the overrides dict below to customise — all other fields use preset defaults.

In [ ]:
# ── Customise here ────────────────────────────────────────────────────────────
OVERRIDES = {
    # Reduce total_tokens for a quick smoke-test (e.g. 100_000_000 = 100M)
    # Set to 10_000_000_000 (10B) for a full overnight run
    "total_tokens": 1_000_000_000,   # 1B tokens ~ 4 hours on 2× T4

    "checkpoint_dir": "/kaggle/working/checkpoints",
    "checkpoint_interval": 500,

    # Optimisation flags — set to False/"standard"/"dense" to ablate
    "optimizer": "muon_adamw",
    "normalize": "ngpt",
    "projection": "monarch",
    "optimizer_bits": 32,      # set to 8 to halve optimizer VRAM (requires bitsandbytes)
    "grad_checkpoint": True,

    "wandb_project": "pcg-llm-tiny-kaggle",
}
# ─────────────────────────────────────────────────────────────────────────────

import sys
sys.path.insert(0, "/kaggle/working/pcg-llm/src")

from pcg_llm.config import TrainingConfig
import json

base = TrainingConfig.from_preset("tiny").to_dict()
base.update(OVERRIDES)

# Convert lists back to tuples for frozen dataclass
for k, v in base.items():
    if k in ("dataset_fineweb_frac", "dataset_stack_frac") and isinstance(v, list):
        base[k] = tuple(v)

config = TrainingConfig(**base)

print(json.dumps(config.to_dict(), indent=2))

---
## 5 — Train (auto-resumes from latest checkpoint)

**On first run**: starts from scratch, writes checkpoints every 500 steps.  
**On restart**: automatically detects the latest valid checkpoint in `checkpoint_dir` and resumes from it.  

Progress is logged every step. W&B link appears after the first log flush (~30 seconds).

> To start from scratch and ignore existing checkpoints, change `resume=True` to `resume=False`.

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

import torch
from pcg_llm.training.trainer import PCGTrainer

torch.manual_seed(42)

trainer = PCGTrainer(config=config)

# Use the HuggingFace streaming dataset when credentials are available;
# fall back to a synthetic token stream for offline/smoke-test runs.
try:
    from pcg_llm.data.streaming import HuggingFaceStreamingDataset
    from pcg_llm.data.tokenizer import Llama3TokenizerWrapper

    tokenizer = Llama3TokenizerWrapper()
    dataset = HuggingFaceStreamingDataset(
        tokenizer=tokenizer,
        seq_len=config.max_seq_len,
        batch_size=config.batch_size,
        seed=42,
    )
    dataloader = dataset
    print("Using HuggingFace streaming dataset (FineWeb-Edu + The Stack v2)")

except Exception as e:
    print(f"HuggingFace dataset unavailable ({e}) — using synthetic token stream")
    from typing import Iterator, Any

    def _synthetic_loader() -> Iterator[Any]:
        while True:
            yield torch.randint(0, config.vocab_size, (config.batch_size, config.max_seq_len))

    dataloader = _synthetic_loader()

trainer.train(dataloader=dataloader, resume=True)

---
## 6 — Inspect checkpoints

In [ ]:
import os
from pathlib import Path

ckpt_dir = Path(config.checkpoint_dir)
checkpoints = sorted(ckpt_dir.glob("step-*.pt"))

print(f"Checkpoints in {ckpt_dir}:")
for ckpt in checkpoints:
    size_mb = ckpt.stat().st_size / 1e6
    print(f"  {ckpt.name}  ({size_mb:.1f} MB)")

manifest_path = ckpt_dir / "manifest.json"
if manifest_path.exists():
    import json
    manifest = json.loads(manifest_path.read_text())
    print(f"\nManifest entries: {len(manifest.get('entries', []))}")
    if manifest.get("entries"):
        latest = manifest["entries"][-1]
        print(f"Latest: step {latest['step']}  sha256={latest['sha256'][:16]}...")

---
## 7 — Quick evaluation (optional)

Runs a lightweight perplexity check on a small held-out batch.  
Full benchmark evaluation (ARC, MMLU, GSM8K, HumanEval) requires `uv sync --extra eval`  
and is better run after the full training run completes.

In [ ]:
import math
import torch
import torch.nn.functional as F

trainer_eval = trainer  # reuse trainer from cell 5 if still in memory

# Generate 4 random held-out batches and compute mean perplexity
total_loss = 0.0
n_batches = 4

with torch.no_grad():
    for _ in range(n_batches):
        batch = torch.randint(
            0, config.vocab_size,
            (config.batch_size, config.max_seq_len),
            device=trainer_eval.device
        )
        metrics = trainer_eval.train_step(batch)
        total_loss += metrics.get("loss", 0.0)

mean_loss = total_loss / n_batches
perplexity = math.exp(mean_loss)

print(f"Mean cross-entropy loss : {mean_loss:.4f}")
print(f"Perplexity              : {perplexity:.1f}")
print()
if perplexity <= 50:
    print("PASS — perplexity is within target (≤ 50)")
else:
    print(f"INFO — perplexity {perplexity:.1f} > 50; more training steps needed")

---
## 8 — Health checks

Key metrics to verify the model is training correctly.

In [ ]:
print("=== Training Health Summary ===")
print(f"Current step          : {trainer.step}")
print(f"Device                : {trainer.device}")

# Solver convergence check
# W&B tracks solver_steps_mean — here we log the config cap for reference
print(f"DEQ max solver iters  : {config.max_solver_iters}")
print(f"  Target: mean steps per token ≤ {config.max_solver_iters * 0.8:.0f} (80% of cap)")

# Sparsity
try:
    sparsity = trainer.adjacency.current_sparsity()
    print(f"Adjacency sparsity    : {sparsity:.2%}")
    print(f"  Target: ≥ {config.initial_sparsity:.0%} (SC-004 lower bound: 80%)")
except Exception:
    print("Adjacency sparsity    : (not available)")

# GPU memory
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"GPU {i} VRAM used      : {alloc:.2f} / {total:.1f} GB  ({alloc/total:.0%})")

# Disk space remaining
import shutil
usage = shutil.disk_usage("/kaggle/working")
free_gb = usage.free / 1e9
print(f"Disk free             : {free_gb:.1f} GB")
if free_gb < 1.0:
    print("  WARNING: less than 1 GB free — checkpointing may fail")

---
## Acceptance Criteria

After a full ~4-hour Tiny PCG run, verify:

| Metric | Target | Where to check |
|--------|--------|----------------|
| Perplexity (held-out 50M tokens) | ≤ 50 | Cell 7 above |
| Mean DEQ solver steps per token | ≤ 8 | W&B `solver_steps_mean` |
| Node variance | > 0.1 throughout | W&B `node_variance` |
| EAGLE draft acceptance rate | ≥ 40% | W&B `eagle_accept_rate` |
| Checkpoints written every 500 steps | ✓ | Cell 6 above |
| Resume from checkpoint without loss spike | ✓ | Restart kernel + rerun cell 5 |

**If solver steps consistently hit the cap (12)**: reduce `base_lr` by 3× (`3e-4`)  
**If node variance collapses to < 0.1**: increase `gamma_variance` to `0.05`  
**If EAGLE acceptance < 40%**: run `trainer.fine_tune_eagle(steps=5000)` after training